## Global SHAP Value calculations:

This notebook analyzes global feature importance for the SensorX MLP fault prediction model using SHAP (SHapley Additive exPlanations) values. 
SHAP values quantify the contribution of each input feature to the model's predictions, helping to interpret which features most influence fault risk. The notebook computes SHAP values for a sample of training data, identifies the top features, and visualizes their impact using summary plots.

In [0]:
%pip install shap -q

In [0]:
import shap
import numpy as np
import mlflow
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.ml.linalg import Vectors
from pyspark.ml.functions import vector_to_array

# Match Plotly default font (same as AUC chart)
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Open Sans', 'Arial', 'Helvetica', 'DejaVu Sans']

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

# Load latest model (v7: 112 features)
model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/7")

# Load training data (112 features, 15-day horizon)
spark.catalog.refreshTable("teams.sensorx.train_data")
train = spark.read.table("teams.sensorx.train_data")

# Feature names: hardcoded from assembler order
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]
dev_cols = [
    "payload_xrayController_filamentCurrent_avg_daily",
    "payload_xrayController_filamentCurrent_dev_daily",
    "payload_xrayController_temperature_avg_daily",
    "payload_xrayController_temperature_dev_daily",
    "payload_xrayController_tubeCurrent_avg_daily",
    "payload_xrayController_tubeCurrent_dev_daily",
]
feature_names = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
short_names = [n.replace("payload_xrayController_", "").replace("_avg_daily", "_avg").replace("_dev_daily", "_dev") for n in feature_names]

# Verify
actual_size = train.select(vector_to_array("features").alias("a")).limit(1).collect()[0]["a"]
assert len(actual_size) == len(feature_names) == 112, f"Mismatch: vector={len(actual_size)}, names={len(feature_names)}"
print(f"Features: {len(feature_names)} (no onTime) | First 5: {short_names[:5]}")

# Sample training data to numpy
N_BG = 500       # background samples
N_EXPLAIN = 500  # samples to explain
N_SAMPLES = 200  # SHAP perturbations

X = np.array(
    train.select(vector_to_array("features").alias("a"))
    .orderBy(F.rand(seed=42)).limit(N_BG + N_EXPLAIN)
    .toPandas()["a"].tolist()
)
background = shap.kmeans(X[:N_BG], 50)
X_explain = X[N_BG:]

# Prediction wrapper
def predict(X):
    rows = [Row(features=Vectors.dense(x.tolist())) for x in X]
    df = spark.createDataFrame(rows)
    probs = model.transform(df).select(vector_to_array("probability").alias("p")).toPandas()
    return np.array(probs["p"].tolist())

# Compute SHAP
print(f"Computing SHAP (N_BG={N_BG}, N_EXPLAIN={N_EXPLAIN}, N_SAMPLES={N_SAMPLES})...")
print("This will take a while with increased samples...")
explainer = shap.KernelExplainer(predict, background)
shap_values = explainer.shap_values(X_explain, nsamples=N_SAMPLES, silent=True)

# Extract class 1 (fault) SHAP values
if isinstance(shap_values, list):
    sv = shap_values[1]
elif shap_values.ndim == 3:
    sv = shap_values[:, :, 1]
else:
    sv = shap_values

# Top 10 features
mean_abs = np.abs(sv).mean(axis=0)
top_idx = np.argsort(mean_abs)[::-1][:10]
print("\nTop 10 global features (MLP v7, 15-day, no onTime):")
for i, idx in enumerate(top_idx, 1):
    print(f"  {i}. {short_names[idx]}: {mean_abs[idx]:.4f}")

# Custom colormap
custom_cmap = LinearSegmentedColormap.from_list(
    "custom_shap", ["#8bbcc4", "#f2ccda", "#e8b898", "#a68ec2"]
)

# SHAP summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_explain, feature_names=short_names, max_display=10, show=False, cmap=custom_cmap)
plt.gcf().set_facecolor("white")
plt.gca().set_facecolor("white")
plt.title("MLP v7 (15-day) \u2014 Global SHAP (112 features)", fontweight="bold")
plt.tight_layout()
plt.show()